In [2]:
!pwd

/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_4/ver_4


In [3]:
%%writefile .env
S3_ENDPOINT=http://localhost:9000
AWS_ACCESS_KEY_ID=minio_admin
AWS_SECRET_ACCESS_KEY=minio_password

Writing .env


In [4]:
%%writefile requirements.txt
Flask==3.0.2
boto3==1.34.49
botocore==1.34.49
werkzeug==3.0.1

Writing requirements.txt


In [5]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 5000

ENV PYTHONUNBUFFERED=1

CMD ["python", "app.py"]

Writing Dockerfile


In [6]:
%%writefile test_app.py
import pytest
from app import app

@pytest.fixture
def client():
    app.config["TESTING"] = True
    with app.test_client() as client:
        yield client

def test_home_route(client):
    response = client.get("/")
    assert response.status_code == 200
    assert b"Diagnostic Web Portal Active" in response.data

Writing test_app.py


In [8]:
!pytest test_app.py -v

============================= test session starts ==============================
platform darwin -- Python 3.11.5, pytest-9.1.1, pluggy-1.6.0 -- /Users/admin/Desktop/Shafer_Python_Classes/pandas_env/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_4/ver_4
plugins: anyio-4.4.0
collected 1 item                                                               

test_app.py::test_home_route PASSED                                      [100%]

============================== 1 passed in 5.68s ===============================


In [ ]:
!docker build -t vehicle-diagnostic-app .

In [10]:
!docker images | grep vehicle-diagnostic-app

vehicle-diagnostic-app                                    latest                                                                       415875b8a11c   12 seconds ago   161MB


In [11]:
!docker run -d --name diagnostic-container -p 5000:5000 vehicle-diagnostic-app

f2d892f654addd03421deb56866c04d973f34589dee83566172ec512373378dd


In [12]:
!docker ps
!docker logs diagnostic-container

CONTAINER ID   IMAGE                    COMMAND                  CREATED             STATUS             PORTS                                            NAMES
f2d892f654ad   vehicle-diagnostic-app   "python app.py"          19 seconds ago      Up 18 seconds      0.0.0.0:5000->5000/tcp                           diagnostic-container
042b375a0051   minio/minio:latest       "/usr/bin/docker-ent…"   About an hour ago   Up About an hour   0.0.0.0:9000->9000/tcp, 0.0.0.0:9004->9001/tcp   minio_ver_tres
65bc50cc6533   minio/minio              "/usr/bin/docker-ent…"   2 hours ago         Up 2 hours                                                          k8s_minio_minio-deployment-557bb94bbf-7g6zw_default_3ed40980-fa39-45b2-8c15-8fb27f9c034c_8
caa4864ab0e2   1734d010a179             "python app.py"          4 hours ago         Up 4 hours                                                          k8s_flask-web_diagnostic-web-deployment-6645c866d8-f79hg_default_71f649bf-0500-41ed-9a10-5b1d3f264b7f_

In [13]:
%%writefile .github/workflows/ci-cd.yml
name: Vehicle Diagnostic CI/CD Pipeline

on:
  push:
    branches: [ "main", "master" ]
  pull_request:
    branches: [ "main", "master" ]

jobs:
  build-and-test:
    runs-on: ubuntu-latest

    steps:
    - name: Checkout Repository
      uses: actions/checkout@v4

    - name: Set up Python 3.11
      uses: actions/setup-python@v5
      with:
        python-version: "3.11"
        cache: 'pip'

    - name: Install Dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements.txt
        pip install pytest

    - name: Run Unit Tests
      run: |
        pytest test_app.py -v

    - name: Set up Docker Buildx
      uses: docker/setup-buildx-action@v3

    - name: Build Docker Image
      uses: docker/build-push-action@v5
      with:
        context: .
        file: ./Dockerfile
        push: false
        tags: vehicle-diagnostic-app:latest

Writing .github/workflows/ci-cd.yml


FileNotFoundError: [Errno 2] No such file or directory: '.github/workflows/ci-cd.yml'

In [14]:
import os
print("Directorio actual:", os.getcwd())

Directorio actual: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_4/ver_4


In [15]:
!pwd

/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_4/ver_4


In [16]:
%%writefile ../../../../../.github/workflows/ci-cd.yml
name: CI/CD Pipeline Workflow

on:
  push:
    branches: [ "main", "master" ]
  pull_request:
    branches: [ "main", "master" ]

jobs:
  build-and-test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout Repository
        uses: actions/checkout@v4

      - name: Set up Python Environment
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

      - name: Execute Tests with Pytest
        run: |
          pytest

Writing ../../../../../.github/workflows/ci-cd.yml


In [17]:
%%writefile .gitignore
# Python
__pycache__/
*.py[cod]
*$py.class
*.so
.Python
env/
venv/
ENV/
pandas_env/

# Jupyter Notebook
.ipynb_checkpoints

# Environment variables / secrets
.env
*.env

# Testing & Coverage
.pytest_cache/
.coverage
htmlcov/

Writing .gitignore


In [18]:
%%writefile .env
FLASK_APP=app.py
FLASK_ENV=development
SECRET_KEY=dev_blueprint_architect_key_phase7
PORT=5000

Overwriting .env


In [19]:
# Ver archivos creados localmente en ver_4
!ls -la .env .gitignore

-rw-r--r--  1 admin  staff   95 Aug  2 17:07 .env
-rw-r--r--  1 admin  staff  225 Aug  2 17:07 .gitignore


In [20]:
# Ver archivo de workflow en la raíz
!ls -la ../../../../../.github/workflows/ci-cd.yml

-rw-r--r--  1 admin  staff  636 Aug  2 17:06 ../../../../../.github/workflows/ci-cd.yml


In [21]:
%%writefile test_app.py
import os
import pytest
from app import app

@pytest.fixture
def client():
    app.config['TESTING'] = True
    with app.test_client() as client:
        yield client

def test_home_status(client):
    """Verifica que el endpoint principal responda 200 OK"""
    response = client.get('/')
    assert response.status_code == 200

def test_environment_variables():
    """Verifica la lectura de configuraciones"""
    # Prueba de lectura de entorno
    assert os.getenv('FLASK_APP', 'app.py') == 'app.py'

Overwriting test_app.py


In [22]:
!pytest -v

============================= test session starts ==============================
platform darwin -- Python 3.11.5, pytest-9.1.1, pluggy-1.6.0 -- /Users/admin/Desktop/Shafer_Python_Classes/pandas_env/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_4/ver_4
plugins: anyio-4.4.0
collected 2 items                                                              

test_app.py::test_home_status PASSED                                     [ 50%]
test_app.py::test_environment_variables PASSED                           [100%]

============================== 2 passed in 0.76s ===============================


In [23]:
# Verificar qué archivos detecta Git localmente
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git checkout -- <file>..." to discard changes in working directory)

	modified:   ../../../../../.github/workflows/main.yml
	modified:   ../../../../../.gitignore
	modified:   ../../../../../Actualizacio_ de_aprendizaje/notes.ipynb
	modified:   ../../../../../Actualizacio_ de_aprendizaje/software_engineering_practices.ipynb
	modified:   ../../../../../Actualizacio_ de_aprendizaje/yaml_y_kubernetes.ipynb
	modified:   ../../../../../Dockerfile
	modified:   ../../../../../Tutorial/Untitled.ipynb
	modified:   ../../../../../Tutorial/app.py
	modified:   ../../../../../Tutorial/requirements.txt
	modified:   ../../../../../Tutorial/static/css/styles.css
	modified:   ../../../../../Tutorial/templates/Untitled.ipynb
	modified:   ../../../../../Tutorial/templates/index.html
	modified:   ../../../../../Tutorial/templates/launch_card.html

In [24]:
# Agregar solo los archivos necesarios ignorando .env
!git add .gitignore .github/workflows/ci-cd.yml app.py test_app.py requirements.txt

fatal: pathspec '.github/workflows/ci-cd.yml' did not match any files


In [25]:
!git add ../../../../../.github/workflows/ci-cd.yml .gitignore .env app.py test_app.py requirements.txt

The following paths are ignored by one of your .gitignore files:
containers/ninth_container/repaso/next_4/ver_4/.env
Use -f if you really want to add them.


In [26]:
!git add ../../../../../.github/workflows/ci-cd.yml .gitignore app.py test_app.py requirements.txt

In [ ]:
!git status

In [28]:
# 1. Crear el commit para la Fase 7
!git commit -m "ci/cd(phase-7): add github actions workflow, pytest suite, and local gitignore"

# 2. Subir los cambios a la rama principal (main o master)
!git push origin main

[main 7922f6f] ci/cd(phase-7): add github actions workflow, pytest suite, and local gitignore
 5 files changed, 234 insertions(+)
 create mode 100644 .github/workflows/ci-cd.yml
 create mode 100644 containers/ninth_container/repaso/next_4/ver_4/.gitignore
 create mode 100644 containers/ninth_container/repaso/next_4/ver_4/app.py
 create mode 100644 containers/ninth_container/repaso/next_4/ver_4/requirements.txt
 create mode 100644 containers/ninth_container/repaso/next_4/ver_4/test_app.py
Counting objects: 13, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (12/12), done.
Writing objects: 100% (13/13), 1.69 KiB | 433.00 KiB/s, done.
Total 13 (delta 5), reused 0 (delta 0)
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/leopinzon75/cloud-engineering-practice.git
   5a8581b..7922f6f  main -> main
